# Polyscope cross-section viewer

Display the mesh, slice planes, contour curves, and signed 2D SDFs generated by `01_data_preparation.ipynb`.

In [4]:
%pip install -q numpy scipy polyscope

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from pathlib import Path

import numpy as np
import polyscope as ps
from scipy.interpolate import RegularGridInterpolator

DATA = Path("data/bunny_cross_sections.npz")
PLANE_RESOLUTION = 90
PLANE_PADDING = 0.12
SDF_LIMIT = 0.35
ACTIVE_SLICE = None
BACKEND = os.environ.get("POLYSCOPE_BACKEND", "auto")

## Load the output of Notebook 01

In [7]:
with np.load(DATA) as archive:
    grid_axis = archive["grid_axis"]
    vertices = archive["vertices"]
    faces = archive["faces"]
    plane_normals = archive["plane_normals"]
    plane_origins = archive["plane_origins"]
    sdf2d = archive["sdf2d"]
    contour_vertices = archive["contour_vertices"]
    contour_edges = archive["contour_edges"]
    contour_plane = archive["contour_plane"]

N_SLICES = len(plane_normals)
print({"mesh": (len(vertices), len(faces)), "slices": N_SLICES, "sdf2d": sdf2d.shape})

{'mesh': (2507, 5010), 'slices': 6, 'sdf2d': (6, 64, 64, 64)}


## Build finite plane meshes

In [9]:
def plane_frame(normal):
    reference = np.eye(3)[np.argmin(np.abs(normal))]
    u = np.cross(normal, reference)
    u /= np.linalg.norm(u)
    v = np.cross(normal, u)
    return np.column_stack((u, v))


def grid_faces(rows, columns):
    row, column = np.meshgrid(np.arange(rows - 1), np.arange(columns - 1), indexing="ij")
    lower_left = (row * columns + column).ravel()
    return np.vstack((
        np.column_stack((lower_left, lower_left + 1, lower_left + columns)),
        np.column_stack((lower_left + 1, lower_left + columns + 1, lower_left + columns)),
    )).astype(np.int32)


def plane_contours(plane_id):
    vertex_ids = np.flatnonzero(contour_plane == plane_id)
    remap = np.full(len(contour_vertices), -1, dtype=np.int32)
    remap[vertex_ids] = np.arange(len(vertex_ids))
    edge_mask = np.isin(contour_edges[:, 0], vertex_ids) & np.isin(contour_edges[:, 1], vertex_ids)
    return contour_vertices[vertex_ids], remap[contour_edges[edge_mask]]


def build_plane(plane_id):
    normal = plane_normals[plane_id]
    origin = plane_origins[plane_id]
    frame = plane_frame(normal)
    curve_vertices, curve_edges = plane_contours(plane_id)
    curve_uv = (curve_vertices - origin) @ frame
    lower = curve_uv.min(axis=0) - PLANE_PADDING
    upper = curve_uv.max(axis=0) + PLANE_PADDING
    u = np.linspace(lower[0], upper[0], PLANE_RESOLUTION)
    v = np.linspace(lower[1], upper[1], PLANE_RESOLUTION)
    uv = np.stack(np.meshgrid(u, v, indexing="xy"), axis=-1)
    plane_vertices = origin + uv.reshape(-1, 2) @ frame.T
    plane_faces = grid_faces(PLANE_RESOLUTION, PLANE_RESOLUTION)
    interpolator = RegularGridInterpolator(
        (grid_axis, grid_axis, grid_axis),
        sdf2d[plane_id],
        bounds_error=False,
        fill_value=None,
    )
    values = interpolator(plane_vertices).astype(np.float32)
    return {
        "vertices": plane_vertices,
        "faces": plane_faces,
        "sdf": values,
        "contour_vertices": curve_vertices,
        "contour_edges": curve_edges,
    }


plane_data = [build_plane(plane_id) for plane_id in range(N_SLICES)]

assert all(np.isfinite(plane["sdf"]).all() for plane in plane_data)
assert all(len(plane["contour_edges"]) > 0 for plane in plane_data)
print([{"slice": i + 1, "vertices": len(plane["vertices"]), "sdf_range": np.round([plane["sdf"].min(), plane["sdf"].max()], 3).tolist()} for i, plane in enumerate(plane_data)])

[{'slice': 1, 'vertices': 8100, 'sdf_range': [-0.5130000114440918, 0.5569999814033508]}, {'slice': 2, 'vertices': 8100, 'sdf_range': [-0.4230000078678131, 0.47099998593330383]}, {'slice': 3, 'vertices': 8100, 'sdf_range': [-0.4950000047683716, 0.5910000205039978]}, {'slice': 4, 'vertices': 8100, 'sdf_range': [-0.45899999141693115, 0.4339999854564667]}, {'slice': 5, 'vertices': 8100, 'sdf_range': [-0.28999999165534973, 0.6970000267028809]}, {'slice': 6, 'vertices': 8100, 'sdf_range': [-0.4390000104904175, 0.43700000643730164]}]


## Register the scene

In [11]:
ps.init(BACKEND)
ps.remove_all_structures()
ps.set_up_dir("y_up")
ps.set_ground_plane_mode("none")
ps.set_transparency_mode("pretty")
ps.set_transparency_render_passes(12)

bunny = ps.register_surface_mesh(
    "bunny",
    vertices,
    faces,
    color=(0.72, 0.72, 0.72),
    smooth_shade=True,
    transparency=0.28,
)

plane_handles = []
contour_handles = []
for plane_id, plane in enumerate(plane_data):
    surface = ps.register_surface_mesh(
        f"slice {plane_id + 1}",
        plane["vertices"],
        plane["faces"],
        smooth_shade=False,
        back_face_policy="identical",
        transparency=0.82,
    )
    surface.add_scalar_quantity(
        "signed 2D SDF",
        plane["sdf"],
        defined_on="vertices",
        datatype="symmetric",
        vminmax=(-SDF_LIMIT, SDF_LIMIT),
        cmap="coolwarm",
        enabled=True,
    )
    contour = ps.register_curve_network(
        f"contour {plane_id + 1}",
        plane["contour_vertices"],
        plane["contour_edges"],
        color=(0.02, 0.02, 0.02),
        radius=0.0035,
    )
    plane_handles.append(surface)
    contour_handles.append(contour)


def show_only(slice_number=None):
    for plane_id, (surface, contour) in enumerate(zip(plane_handles, contour_handles), start=1):
        enabled = slice_number is None or plane_id == slice_number
        surface.set_enabled(enabled)
        contour.set_enabled(enabled)


show_only(ACTIVE_SLICE)

## Open Polyscope

Set `ACTIVE_SLICE` above to `1` through `N_SLICES` to isolate one plane, or leave it as `None` to show all planes. The Polyscope sidebar can also toggle every slice and contour independently.

In [13]:
ps.show(1 if BACKEND == "openGL_mock" else None)